# Nueva York en movimiento
## Taxis amarillos y clima, 2024
EDA reproducible de dos fuentes independientes: NYC TLC y NOAA. El dataset final supera los minimos del master. Todas las salidas de codigo incluidas se han ejecutado sobre los archivos entregados.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
df = pd.read_parquet(ROOT / 'data/processed/taxi_clima_final.parquet')
print('Dimensiones:', df.shape)
print('Fechas UTC:', df.fecha_utc.min(), df.fecha_utc.max())
print(df[['fecha_utc','duracion_min','distancia_km','fare_amount','lluvia']].head().to_string(index=False))

Dimensiones: (576606, 44)
Fechas UTC: 2024-01-02 2024-12-31
 fecha_utc  duracion_min  distancia_km  fare_amount        lluvia
2024-01-02      6.166667      2.076054        36.83 Sin indicador
2024-01-02      3.233333      1.384036         5.80 Sin indicador
2024-01-02     17.783333     15.031273        36.60 Sin indicador
2024-01-02      9.000000      1.657624        10.00 Sin indicador
2024-01-02     13.766667      2.365736        13.50 Sin indicador


## 1. Fuentes y diseno muestral
Se toman 50.000 filas sin limpiar por mes, sin reemplazo, usando semilla 202400+mes. Los valores fuente se conservan en el extracto bruto. Las dos columnas anadidas permiten localizar la fila de origen. El peso es N del mes / 50.000. Data.gov es el catalogo, TLC y NOAA son los productores independientes.

In [2]:
manifest = json.loads((ROOT/'data/raw/manifiesto_tlc.json').read_text())
print(pd.DataFrame(manifest)[['mes','filas_originales','filas_extraidas','semilla']].to_string(index=False))
print('Total fuente:', sum(x['filas_originales'] for x in manifest))

 mes  filas_originales  filas_extraidas  semilla
   1           2964624            50000   202401
   2           3007526            50000   202402
   3           3582628            50000   202403
   4           3514289            50000   202404
   5           3723833            50000   202405
   6           3539193            50000   202406
   7           3076903            50000   202407
   8           2979183            50000   202408
   9           3633030            50000   202409
  10           3833771            50000   202410
  11           3646369            50000   202411
  12           3668371            50000   202412
Total fuente: 41169720


## 2. Limpieza y controles
Las reglas estan implementadas en src/01_preparar.py. No se imputan pasajeros o precipitacion ausentes. Las horas ambiguas del cambio de horario se excluyen. Se conservan unicamente dias UTC completos. La union se valida como muchos-a-uno, sin multiplicar viajes.

In [3]:
quality = json.loads((ROOT/'reports/validacion_final.json').read_text(encoding='utf-8'))
print(json.dumps(quality,ensure_ascii=False,indent=2))
assert len(df)>=50000 and len(df.columns)>=20
assert not df.duplicated(['mes_archivo','fila_origen']).any()
assert df.estacion_meteo.notna().all()
print(pd.read_csv(ROOT/'reports/calidad_datos.csv').query('ausentes > 0').to_string(index=False))

{
  "filas_brutas": 600000,
  "filas_finales": 576606,
  "columnas_finales": 44,
  "dias_utc_completos": 365,
  "meses_archivo": 12,
  "sin_correspondencia_meteo": 0,
  "horas_DST_ambiguas_o_inexistentes_excluidas": 120,
  "filas_dias_UTC_parciales_excluidas": 1484,
  "coincidencias_exactas_conservadas": 0,
  "registros_fuente_TLC": 41169720,
  "reglas": [
    {
      "regla": "Fecha fuera del mes y ano de origen",
      "incidencias": 2,
      "exclusiones_adicionales": 2
    },
    {
      "regla": "Duracion no positiva, ausente o superior a 180 min",
      "incidencias": 592,
      "exclusiones_adicionales": 592
    },
    {
      "regla": "Distancia no positiva, ausente o superior a 100 millas",
      "incidencias": 11455,
      "exclusiones_adicionales": 11270
    },
    {
      "regla": "Tarifa o importe no positivo",
      "incidencias": 10666,
      "exclusiones_adicionales": 9789
    },
    {
      "regla": "Propina negativa o ausente",
      "incidencias": 16,
      "exclusio

## 3. EDA ponderado
Los meses tienen el mismo tamano muestral, pero distinto numero de registros fuente. Por eso las medias, porcentajes, totales y cuantiles del informe utilizan pesos. La proporcion de propina agregada es una razon de sumas, no la media de porcentajes por viaje.

In [4]:
w = df.peso_muestral
for col in ['duracion_min','distancia_km','fare_amount','total_amount']:
    print(col, 'media ponderada =', round(np.average(df[col],weights=w),3))
card = df[df.payment_type.eq(1)]
print('Propina / tarifa tarjeta (%) =',100*(card.tip_amount*card.peso_muestral).sum()/(card.fare_amount*card.peso_muestral).sum())
print('Volumen valido estimado =',w.sum())

duracion_min media ponderada = 16.941
distancia_km media ponderada = 5.517
fare_amount media ponderada = 19.817
total_amount media ponderada = 28.671
Propina / tarifa tarjeta (%) = 22.092060360126137
Volumen valido estimado = 39571896.90992


## 4. Tabla diaria e independencia
Cada fila de esta tabla es un dia UTC completo. Para estudiar el clima, 576.606 viajes no equivalen a 576.606 observaciones meteorologicas independientes. Hay 365 dias, con 138 dias de lluvia indicada y 227 sin indicador. El cero del indicador NOAA puede significar no o no reportado.

In [5]:
daily = pd.read_csv(ROOT/'data/processed/resumen_diario.csv')
print(daily[['fecha_utc','viajes_estimados','duracion_media','lluvia_indicada']].head().to_string(index=False))
print('Dias:',len(daily))
print(daily.lluvia_indicada.value_counts().to_string())

 fecha_utc  viajes_estimados  duracion_media  lluvia_indicada
2024-01-02       65992.53024       16.155720            False
2024-01-03       78621.82848       15.341830            False
2024-01-04       89650.22976       15.962169            False
2024-01-05      100737.92352       14.732166            False
2024-01-06       93326.36352       13.825011            False
Dias: 365
lluvia_indicada
False    227
True     138


## 5. Modelo ajustado de actividad
Se ajusta log(volumen diario estimado) por lluvia, temperatura, nieve, mes y dia semanal UTC. Los errores HAC con siete retardos contemplan dependencia temporal semanal y heterocedasticidad. Es asociacion observacional, no efecto causal.

In [6]:
import statsmodels.formula.api as smf
model = smf.ols('np.log(viajes_estimados) ~ lluvia_num + temperatura_c + nieve_num + C(mes) + C(dia_semana)',data=daily).fit(cov_type='HAC',cov_kwds={'maxlags':7})
print(model.summary().as_text())
print('Cambio ajustado por lluvia (%):',100*np.expm1(model.params['lluvia_num']))

                               OLS Regression Results                               
Dep. Variable:     np.log(viajes_estimados)   R-squared:                       0.488
Model:                                  OLS   Adj. R-squared:                  0.459
Method:                       Least Squares   F-statistic:                     33.82
Date:                      Fri, 06 Nov 2026   Prob (F-statistic):           7.27e-69
Time:                              15:37:01   Log-Likelihood:                 237.39
No. Observations:                       365   AIC:                            -432.8
Df Residuals:                           344   BIC:                            -350.9
Df Model:                                20                                         
Covariance Type:                        HAC                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------

## 6. Tres preguntas y correccion de Holm
Se ajustan tres modelos diarios: volumen, duracion (tambien ajustada por distancia media) y razon de propina sobre tarifa en tarjeta. Se corrigen los tres p-valores de lluvia con Holm. Los intervalos del informe son puntuales del 95%, no simultaneos corregidos.

In [7]:
results = pd.read_csv(ROOT/'reports/resultados_estadisticos.csv')
print(results[['modelo','efecto','ic95_inferior','ic95_superior','p_holm','n_dias']].to_string(index=False))
assert (results.n_dias==365).all()
print('Ninguna prueba tiene p Holm < 0.05:', (results.p_holm>=.05).all())

                               modelo   efecto  ic95_inferior  ic95_superior   p_holm  n_dias
              Volumen diario estimado 2.580636      -0.361251       5.609384 0.258379     365
                Duracion media diaria 0.903023      -0.420573       2.244213 0.364162     365
Propina diaria sobre tarifa (tarjeta) 0.069539      -0.036856       0.175934 0.364162     365
Ninguna prueba tiene p Holm < 0.05: True


## 7. Visualizaciones e interpretacion
Las siete figuras estan en reports/figuras y en la memoria PDF. Los graficos horarios comparan viajes medios por dia, no totales de grupos con distinto numero de dias. Los histogramas recortan solo el eje mostrado. El dashboard muestra comparaciones descriptivas; no sustituye los modelos ajustados.

![Actividad mensual](../reports/figuras/01_actividad_mensual.png)
![Perfil horario](../reports/figuras/04_perfil_horario.png)
![Modelos](../reports/figuras/06_modelos.png)

## 8. Sensibilidad, conclusiones y limites
Al reducir el maximo de duracion a 120 minutos, la estimacion de volumen asociada con lluvia pasa de 2,58% a 2,57%. Las asociaciones no alcanzan significacion al 5%; no se demuestra ausencia de efecto. Una estacion y un indicador diario limitan la medicion. No se incluyen festivos o disponibilidad de taxis y el volumen es una estimacion muestral.

Para reconstruir todos los resultados: ejecutar src/01_preparar.py y src/02_analizar.py. Para reconstruir la muestra desde las fuentes completas: src/00_descargar.py. La memoria detalla la metodologia y las referencias.